## 1. Install and verify the training environment

Run before importing Paddle, NumPy, or OpenCV. If those were already imported, restart the runtime once and rerun from this cell. The training source is pinned to PaddleOCR v3.7.0. The download/install cell checks for a real GPU and does not silently fall back to CPU.


In [ ]:
import os
import sys
import subprocess
import urllib.request
import zipfile
import shutil
from pathlib import Path

PADDLE_VERSION = "3.3.0"
PADDLE_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu118/"
PADDLEOCR_TAG = "v3.7.0"
REPO_DIR = Path("/content/PaddleOCR-3.7.0")
ARCHIVE = Path("/content/PaddleOCR-v3.7.0.zip")

def download_atomic(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".part")
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "digit-ocr-colab/1.0"})
        with urllib.request.urlopen(request, timeout=120) as response, temporary.open("wb") as stream:
            shutil.copyfileobj(response, stream)
        if temporary.stat().st_size == 0:
            raise RuntimeError("The download was empty.")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)

if not (REPO_DIR / "tools/train.py").exists():
    download_atomic(
        f"https://github.com/PaddlePaddle/PaddleOCR/archive/refs/tags/{PADDLEOCR_TAG}.zip",
        ARCHIVE,
    )
    with zipfile.ZipFile(ARCHIVE) as archive:
        archive.extractall("/content")

subprocess.run([sys.executable, "-m", "pip", "install", f"paddlepaddle-gpu=={PADDLE_VERSION}",
                "--index-url", PADDLE_INDEX], check=True)
requirements = []
for line in (REPO_DIR / "requirements.txt").read_text().splitlines():
    value = line.strip()
    if value and not value.startswith("#") and not value.lower().startswith(
        ("opencv-", "numpy", "albumentations", "albucore")
    ):
        requirements.append(value)
subprocess.run([sys.executable, "-m", "pip", "install", *requirements,
                "numpy==2.1.3", "albumentations==2.0.8", "matplotlib>=3.9,<4",
                "huggingface_hub==0.36.0", "datasets==3.6.0", "pyarrow==22.0.0", "Pillow>=10.4"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "opencv-python",
                "opencv-contrib-python", "opencv-python-headless", "opencv-contrib-python-headless"],
               check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
                "--no-deps", "opencv-python-headless==4.12.0.88"], check=True)
subprocess.run([sys.executable, "-c", "import paddle, cv2, numpy, albumentations; "
                "print('Paddle:', paddle.__version__); print('OpenCV:', cv2.__version__); "
                "print('NumPy:', numpy.__version__); "
                "assert paddle.is_compiled_with_cuda(), 'A GPU Paddle build is required'; "
                "assert paddle.device.cuda.device_count() > 0, 'Select a GPU runtime'"], check=True)

## 2. Settings

Defaults match the naming convention you supplied: **`0001_0000.jpg` means label `0000`**, not `0001`.

- `PERSONAL_LABEL_MODE="suffix"`: the last underscore-separated component is the label.
- Use `"prefix"` only for a verified convention such as `0145_2.jpg` meaning `0145`.
- `labels.tsv`, if present, overrides filename-derived labels.
- `LABEL_POLICY="digits_only"`: keep leading zeros, ignore decimal dots, reject negative values. This matches the earlier digits-only project. A display reading `03.42` has the target `0342` under this policy.
- To learn the actual decimal/minus characters, use `LABEL_POLICY="as_written"` **before the first run**. The vocabulary then becomes `0123456789.-`, and your personal labels must include those characters correctly too.
- Every source example is downloaded, but examples incompatible with the chosen label policy are excluded with a reason. The notebook prints exact accepted/rejected counts; it does not claim all 3,333 examples entered training.

`PERSONAL_SPLIT_BY="label"` preserves the original unseen-number split rule. If your manifest has device/session/video IDs, choose `"group"` instead and use a new `SPLIT_STATE_PATH`. Do not change the split while comparing runs.

Checkpoints and reports are preserved in Drive. Training image copies, the Hugging Face cache, export files, and the downloadable ZIP use temporary `/content` storage.


In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
import copy
import csv
import hashlib
import io
import json
import math
import random
import re
from collections import Counter, defaultdict

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/digit_ocr_v6")
PERSONAL_DATA_DIR = PROJECT_DIR / "images"
PERSONAL_MANIFEST = PERSONAL_DATA_DIR / "labels.tsv"
PERSONAL_LABEL_MODE = "suffix"
PERSONAL_SPLIT_BY = "label"
LABEL_POLICY = "digits_only"
SEED = 42
IMAGE_HEIGHT = 64
IMAGE_WIDTH = 320
MAX_TEXT_LENGTH = 25
BATCH_SIZE = 32
NUM_WORKERS = 2
MIXED_EPOCHS = 35
PERSONAL_ADAPT_EPOCHS = 15
MIXED_LEARNING_RATE = 0.0002
PERSONAL_LEARNING_RATE = 0.00005
TARGET_PERSONAL_FRACTION = 0.50
AUGMENTATION_ENABLED = True
INCLUDE_BP_DIGITS = True
INCLUDE_RENJITH = True
FIXED_CROP_NORMALIZED = None

HF_REPO = "MiXaiLL76/7SEG_OCR"
HF_REVISION = "59873e636d5bd8bd850b3ba5788ee761cef5e895"
HF_PARQUET_FILE = "data/train-00000-of-00001.parquet"
HF_EXPECTED_ROWS = 3333
BP_REPO = "chakibmed/bp-monitor-seven-segment"
RENJITH_REPO_URL = "https://github.com/renjithsasidharan/seven-segment-ocr.git"

RUN_NAME = "medium_mixed_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_DIR = PROJECT_DIR / "runs" / RUN_NAME
LOCAL_DIR = Path("/content/digit_ocr_mixed") / RUN_NAME
PREPARED_DIR = LOCAL_DIR / "images"
LIST_DIR = RUN_DIR / "lists"
MODEL_CACHE = PROJECT_DIR / "models"
SPLIT_STATE_PATH = PROJECT_DIR / "split_assignments_medium.json"
CHARACTERS = "0123456789" if LABEL_POLICY == "digits_only" else "0123456789.-"

assert LABEL_POLICY in {"digits_only", "as_written"}
assert PERSONAL_LABEL_MODE in {"suffix", "prefix", "strict"}
assert PERSONAL_SPLIT_BY in {"label", "group"}
assert 0 < TARGET_PERSONAL_FRACTION < 1
assert IMAGE_HEIGHT % 16 == 0 and IMAGE_WIDTH % 16 == 0
assert MIXED_EPOCHS >= 1 and PERSONAL_ADAPT_EPOCHS >= 0
for directory in [PERSONAL_DATA_DIR, RUN_DIR, LOCAL_DIR, PREPARED_DIR, LIST_DIR, MODEL_CACHE]:
    directory.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
print("Run:", RUN_DIR)
print("Label convention:", PERSONAL_LABEL_MODE)
print("Label policy:", LABEL_POLICY, "Vocabulary:", repr(CHARACTERS))
print("Input shape:", [3, IMAGE_HEIGHT, IMAGE_WIDTH])


## 3. Optional personal-data upload

If your existing Drive image folder is ready, leave `UPLOAD_PERSONAL_DATA=False`.

An optional `labels.tsv` at the image-folder root must have tab-separated `path` and `label` headers. Optional `group` values identify a camera/session/video. Optional `x0`, `y0`, `x1`, `y1` columns provide a pixel crop in the EXIF-corrected image. One row must describe exactly one target display row. The manifest defines the complete personal dataset; unlisted files are not used.

```text
path	label	group
camera_a/0001_0000.jpg	0000	camera_a_session_1
camera_a/0002_0145.jpg	0145	camera_a_session_1
```

Uploads never silently overwrite existing files with different bytes. To intentionally replace a manifest, edit it in Drive before preparing the data.


In [ ]:
import hashlib
import io
from pathlib import PurePosixPath
from google.colab import files

UPLOAD_PERSONAL_DATA = False

def write_uploaded_member(relative_name, content):
    name = PurePosixPath(relative_name.replace("\\", "/"))
    if name.is_absolute() or ".." in name.parts or not name.parts or ":" in name.parts[0]:
        raise ValueError(f"Unsafe archive path: {relative_name}")
    target = PERSONAL_DATA_DIR.joinpath(*name.parts)
    if target.is_symlink():
        raise ValueError(f"Symbolic links are not accepted: {target}")
    if not target.resolve().is_relative_to(PERSONAL_DATA_DIR.resolve()):
        raise ValueError(f"Path escapes the data folder: {target}")
    if target.exists():
        if hashlib.sha256(target.read_bytes()).digest() != hashlib.sha256(content).digest():
            raise FileExistsError(f"Conflicting file already exists: {target}")
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)

if UPLOAD_PERSONAL_DATA:
    uploaded = files.upload()
    for name, content in uploaded.items():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(content)) as archive:
                members = [item for item in archive.infolist() if not item.is_dir()]
                if sum(item.file_size for item in members) > 8 * 1024**3:
                    raise ValueError("The uncompressed archive exceeds 8 GiB.")
                for item in members:
                    if "__MACOSX" not in PurePosixPath(item.filename).parts:
                        write_uploaded_member(item.filename, archive.read(item))
        else:
            write_uploaded_member(Path(name).name, content)
print("Data folder:", PERSONAL_DATA_DIR)
print("Manifest found:", PERSONAL_MANIFEST.exists())

## 4. Shared preparation and label rules

These functions are used for both datasets. No random crop or pixel thresholding is applied. A crop that is wider than the configured aspect ratio receives vertical border padding so subsequent resize does not squeeze the digits. Your fixed/per-image crops apply only to your personal images, never to unrelated synthetic images.


In [ ]:
import numpy as np
import cv2
from PIL import Image, ImageOps

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def normalize_label(raw):
    text = str(raw).strip()
    if not text:
        raise ValueError("empty_label")
    if LABEL_POLICY == "digits_only":
        if "-" in text:
            raise ValueError("negative_excluded_by_policy")
        if set(text) - set("0123456789."):
            raise ValueError("unsupported_characters")
        text = text.replace(".", "")
    elif set(text) - set(CHARACTERS):
        raise ValueError("unsupported_characters")
    if not text or not any(c.isdigit() for c in text):
        raise ValueError("no_digits")
    if len(text) > MAX_TEXT_LENGTH:
        raise ValueError("label_too_long")
    return text

def filename_label(path):
    stem = Path(path).stem
    if PERSONAL_LABEL_MODE == "suffix":
        return stem.rsplit("_", 1)[-1]
    if PERSONAL_LABEL_MODE == "prefix":
        return stem.split("_", 1)[0]
    if "_" in stem:
        raise ValueError("Ambiguous filename; use labels.tsv or an explicit filename rule.")
    return stem

def personal_crop(image, row):
    values = [str(row.get(key) or "").strip() for key in ("x0", "y0", "x1", "y1")]
    if any(values):
        if not all(values):
            raise ValueError("All four crop coordinates are required.")
        box = tuple(int(value) for value in values)
    elif FIXED_CROP_NORMALIZED is not None:
        x0, y0, x1, y1 = FIXED_CROP_NORMALIZED
        if not (0 <= x0 < x1 <= 1 and 0 <= y0 < y1 <= 1):
            raise ValueError("Normalized crop coordinates must be inside [0, 1].")
        box = (round(x0 * image.width), round(y0 * image.height),
               round(x1 * image.width), round(y1 * image.height))
    else:
        return image
    x0, y0, x1, y1 = box
    if not (0 <= x0 < x1 <= image.width and 0 <= y0 < y1 <= image.height):
        raise ValueError(f"Invalid crop {box} for image size {image.size}.")
    return image.crop(box)

def fit_input_aspect(image):
    required_height = math.ceil(image.width * IMAGE_HEIGHT / IMAGE_WIDTH)
    extra = max(0, required_height - image.height)
    if extra:
        array = np.asarray(image)
        array = cv2.copyMakeBorder(array, extra // 2, extra - extra // 2, 0, 0, cv2.BORDER_REPLICATE)
        image = Image.fromarray(array)
    return image, extra

def image_digest(image):
    return hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()

def save_prepared(image, origin, source, label, original_label, group="", split="train"):
    if min(image.size) < 8:
        raise ValueError(f"Image crop is too small: {image.size}.")
    digest = image_digest(image)
    fitted, padding = fit_input_aspect(image)
    destination = PREPARED_DIR / f"{digest}.png"
    fitted.save(destination)
    return {"path": destination.name, "origin": origin, "source": source,
            "label": label, "original_label": str(original_label), "group": group,
            "split": split, "sha256": digest, "width": fitted.width, "height": fitted.height,
            "vertical_padding": padding}

def dump_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def write_label_list(path, items):
    Path(path).write_text("".join(f"{item['path']}\t{item['label']}\n" for item in items), encoding="utf-8")

def load_personal_rows():
    if PERSONAL_MANIFEST.exists():
        with PERSONAL_MANIFEST.open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream, delimiter="\t")
            if not {"path", "label"}.issubset(set(reader.fieldnames or [])):
                raise ValueError("labels.tsv requires path and label columns.")
            return list(reader)
    return [{"path": path.relative_to(PERSONAL_DATA_DIR).as_posix(), "label": filename_label(path)}
            for path in sorted(PERSONAL_DATA_DIR.rglob("*"))
            if path.suffix.lower() in IMAGE_EXTENSIONS and "__MACOSX" not in path.parts]

## 5. Prepare personal data and preserve the hold-out split

Your personal data is validated first. Corrupt images, incorrect crops, duplicate paths, and conflicting labels stop the run with a report. Exact duplicate images with the same label are kept once. All versions of an image from one session should share a manifest `group` when session separation is required.

The split state is compatible with the previous medium notebook when seed and split mode are unchanged. The original label-hash rule is retained. A label-policy change is rejected when the split file already records a different policy.


In [ ]:
rows = load_personal_rows()
if not rows:
    raise ValueError(f"No personal images found in {PERSONAL_DATA_DIR}.")
personal_records, personal_errors, duplicate_report = [], [], []
seen_pixels, seen_paths = {}, set()
for index, row in enumerate(rows, 1):
    try:
        relative = str(row["path"]).strip()
        source = (PERSONAL_DATA_DIR / relative).resolve()
        if not source.is_relative_to(PERSONAL_DATA_DIR.resolve()):
            raise ValueError("Image path escapes the personal data folder.")
        if source.suffix.lower() not in IMAGE_EXTENSIONS:
            raise ValueError("Unsupported image extension.")
        if relative in seen_paths:
            raise ValueError("The manifest contains a repeated image path.")
        seen_paths.add(relative)
        label = normalize_label(row["label"])
        group = str(row.get("group") or "").strip()
        if PERSONAL_SPLIT_BY == "group" and not group:
            raise ValueError("Every personal row needs a group for group splitting.")
        with Image.open(source) as original:
            image = personal_crop(ImageOps.exif_transpose(original).convert("RGB"), row)
        digest = image_digest(image)
        if digest in seen_pixels:
            kept = seen_pixels[digest]
            if kept["label"] != label:
                raise ValueError(f"Identical pixels have conflicting labels: {kept['source']} and {relative}.")
            duplicate_report.append({"origin": "personal", "source": relative, "kept": kept["source"]})
            continue
        item = save_prepared(image, "personal", relative, label, row["label"], group)
        seen_pixels[digest] = item
        personal_records.append(item)
    except Exception as error:
        personal_errors.append({"row": index, "source": row.get("path", ""), "error": str(error)})
dump_json(RUN_DIR / "personal_errors.json", personal_errors)
if personal_errors:
    print(json.dumps(personal_errors[:20], indent=2))
    raise ValueError(f"Fix {len(personal_errors)} personal-data errors before continuing.")

if SPLIT_STATE_PATH.exists():
    split_state = json.loads(SPLIT_STATE_PATH.read_text())
    if split_state["seed"] != SEED or split_state["split_by"] != PERSONAL_SPLIT_BY:
        raise ValueError("Split seed/mode changed. Intentionally select a new split-state file.")
    if split_state.get("label_policy", LABEL_POLICY) != LABEL_POLICY:
        raise ValueError("The saved split uses a different label policy. Select a new split-state file.")
else:
    split_state = {"seed": SEED, "split_by": PERSONAL_SPLIT_BY, "assignments": {}}
split_state["label_policy"] = LABEL_POLICY
assignments = split_state["assignments"]
for item in personal_records:
    key = item["label"] if PERSONAL_SPLIT_BY == "label" else item["group"]
    if key not in assignments:
        bucket = int(hashlib.sha1(f"{SEED}:{key}".encode()).hexdigest(), 16) % 100
        assignments[key] = "train" if bucket < 80 else "val" if bucket < 90 else "test"
    item["split"] = assignments[key]
personal_splits = {name: [r for r in personal_records if r["split"] == name]
                   for name in ("train", "val", "test")}
if any(not value for value in personal_splits.values()):
    raise ValueError("A personal split is empty. Add independent groups or establish a new split policy.")
train_characters = set("".join(r["label"] for r in personal_splits["train"]))
held_characters = set("".join(r["label"] for r in personal_records if r["split"] != "train"))
if held_characters - train_characters:
    raise ValueError(f"Held-out characters missing from personal training: {held_characters - train_characters}.")
temporary = SPLIT_STATE_PATH.with_suffix(".tmp")
dump_json(temporary, split_state)
temporary.replace(SPLIT_STATE_PATH)
dump_json(RUN_DIR / "personal_manifest.json", personal_records)
print("Personal source rows:", len(rows), "Unique prepared images:", len(personal_records))
print("Personal split counts:", {name: len(value) for name, value in personal_splits.items()})
print("Personal training digit counts:", dict(Counter("".join(r["label"] for r in personal_splits["train"]))))
print("Training zero-only strings:", sum(set(r["label"]) == {"0"} for r in personal_splits["train"]))

## 6. Download and merge the pinned public dataset

The original Parquet is downloaded through Hugging Face Hub and read in small Arrow batches. No Hugging Face login or token is required for this public revision. No dataset script or remote code is executed.

Public data goes into **training only**. Exact pixel duplicates of any personal image are excluded. With label-based personal splitting, public examples whose normalized text equals a personal validation/test label are also excluded, preserving the unseen-number policy.

The license and attribution are included in the final package. Label-policy exclusions are counted separately from decoding errors; image decoding errors stop the run.


In [ ]:
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

hf_cache = Path("/content/hf_digit_ocr_cache")
hf_parquet = Path(hf_hub_download(
    repo_id=HF_REPO, repo_type="dataset", filename=HF_PARQUET_FILE,
    revision=HF_REVISION, cache_dir=str(hf_cache), token=False, etag_timeout=30,
))
hf_readme = Path(hf_hub_download(
    repo_id=HF_REPO, repo_type="dataset", filename="README.md",
    revision=HF_REVISION, cache_dir=str(hf_cache), token=False, etag_timeout=30,
))
shutil.copy2(hf_readme, RUN_DIR / "7SEG_OCR_README.md")
parquet = pq.ParquetFile(hf_parquet)
if parquet.metadata.num_rows != HF_EXPECTED_ROWS:
    raise ValueError(f"Unexpected source row count at pinned revision: {parquet.metadata.num_rows}.")
if not {"image", "text"}.issubset(set(parquet.schema_arrow.names)):
    raise ValueError("The public dataset schema must contain image and text.")
held_labels = {r["label"] for r in personal_records if r["split"] != "train"}
public_records, public_rejections, public_errors = [], [], []
source_index = 0
for batch in parquet.iter_batches(batch_size=32, columns=["image", "text"]):
    for row in batch.to_pylist():
        index = source_index
        source_index += 1
        source_id = f"{HF_REPO}@{HF_REVISION}:{index}"
        try:
            label = normalize_label(row["text"])
        except ValueError as error:
            public_rejections.append({"row": index, "text": row["text"], "reason": str(error)})
            continue
        if PERSONAL_SPLIT_BY == "label" and label in held_labels:
            public_rejections.append({"row": index, "text": row["text"], "reason": "held_out_personal_label"})
            continue
        try:
            value = row["image"]
            if not isinstance(value, dict) or not value.get("bytes"):
                raise ValueError("An embedded image byte payload was expected.")
            with Image.open(io.BytesIO(value["bytes"])) as original:
                image = ImageOps.exif_transpose(original).convert("RGB")
            digest = image_digest(image)
            if digest in seen_pixels:
                public_rejections.append({"row": index, "text": row["text"], "reason": "duplicate_pixels"})
                if seen_pixels[digest]["label"] != label:
                    raise ValueError("Duplicate pixels have conflicting labels across sources.")
                continue
            item = save_prepared(image, "7SEG_OCR", source_id, label, row["text"], split="train")
            seen_pixels[digest] = item
            public_records.append(item)
        except Exception as error:
            public_errors.append({"row": index, "text": row["text"], "error": str(error)})
dump_json(RUN_DIR / "public_rejections.json", public_rejections)
dump_json(RUN_DIR / "public_errors.json", public_errors)
if public_errors:
    print(json.dumps(public_errors[:10], indent=2))
    raise ValueError("Fix public-image errors before training; no invalid image was silently substituted.")
if not public_records:
    raise ValueError("No compatible public examples remain after filtering.")
records = personal_records + public_records
dump_json(RUN_DIR / "public_manifest.json", public_records)
dump_json(RUN_DIR / "prepared_manifest.json", records)
dump_json(RUN_DIR / "duplicate_report.json", duplicate_report)
print("Downloaded public rows:", source_index)
print("Accepted public training images:", len(public_records))
print("Public exclusions:", dict(Counter(r["reason"] for r in public_rejections)))
print("Accepted public label lengths:", dict(sorted(Counter(len(r["label"]) for r in public_records).items())))
print("External validation/test images:", sum(r["origin"] != "personal" and r["split"] != "train" for r in records))

## 6b. Add two more recognition datasets (training only)

**Blood-pressure dataset:** The `readings` configuration contains 55 **three-row** screen images with separate `systolic`, `diastolic`, and `pulse` fields. Treating one full screen image as a single-number label would corrupt OCR training. We instead use all 388 already-labelled **single-digit** crops from the `digits` configuration (`dev` and `test`), with their actual digit label. They are added to **train only**; the original dataset's test split must no longer be reported as an independent benchmark after this training.

**Renjith dataset:** Clone the GitHub repository and read `data/ocr_training_data/gt.txt` (filename, TAB, full recognized string); these are already recognition crops, not YOLO boxes. The notebook uses the same `normalize_label` policy, checks every file, prevents pixel duplicates and personal-label split leakage, and saves source-specific manifests. The shared `public_records` list drives the existing mixed training stage.

These data sources are **optional** via `INCLUDE_BP_DIGITS` and `INCLUDE_RENJITH`. Downloads only happen in Colab. If access fails, the cell stops instead of pretending data were included.

In [ ]:
import numpy as np
import contextlib

@contextlib.contextmanager
def _sevenseg_nep50_compat():
    yield

np._no_nep50_warning = _sevenseg_nep50_compat

# Test context manager
with np._no_nep50_warning():
    pass

# Test decorator
@np._no_nep50_warning()
def test_numpy_compat():
    return True

assert test_numpy_compat()

print("NumPy compatibility fixed!")

from datasets import load_dataset

extra_source_rejections = []
extra_source_errors = []
extra_source_stats = {}

def accept_external_image(image, raw_label, origin, source_id, record_index, group=""):
    """Validates text, overlap and pixels; never synthesizes an OCR label."""
    try:
        label = normalize_label(raw_label)
    except ValueError as exc:
        extra_source_rejections.append({"origin": origin, "row": record_index,
                                        "source": source_id, "reason": str(exc)})
        return False
    if PERSONAL_SPLIT_BY == "label" and label in held_labels:
        extra_source_rejections.append({"origin": origin, "row": record_index,
                                        "source": source_id, "reason": "held_out_personal_label"})
        return False
    try:
        if not isinstance(image, Image.Image):
            raise ValueError("Expected a PIL image")
        rgb = ImageOps.exif_transpose(image).convert("RGB")
        digest = image_digest(rgb)
        if digest in seen_pixels:
            prior = seen_pixels[digest]
            if prior["label"] != label:
                raise ValueError(f"identical_pixels_conflicting_labels: {prior['label']} != {label}")
            extra_source_rejections.append({"origin": origin, "row": record_index,
                                            "source": source_id, "reason": "duplicate_pixels"})
            return False
        item = save_prepared(rgb, origin, source_id, label, raw_label,
                             group=group, split="train")
        seen_pixels[digest] = item
        public_records.append(item)
        return True
    except Exception as exc:
        extra_source_errors.append({"origin": origin, "row": record_index,
                                    "source": source_id, "error": str(exc)})
        return False

if INCLUDE_BP_DIGITS:
    # Each digit image has exactly one truthful OCR character label.
    bp_data = load_dataset(BP_REPO, "digits", cache_dir="/content/hf_bp_digits_cache")
    bp_counts = {}
    for split_name, split_data in bp_data.items():
        accepted = 0
        for index, row in enumerate(split_data):
            label = str(int(row["label"]))
            accepted += int(accept_external_image(
                row["image"], label, "bp_monitor_digits",
                f"{BP_REPO}:{split_name}:{row['reading_id']}:{row['row']}:{row['position']}",
                index, group=str(row["reading_id"])))
        bp_counts[split_name] = {"provided": len(split_data), "accepted": accepted}
    extra_source_stats["bp_monitor_digits"] = bp_counts
    assert sum(len(part) for part in bp_data.values()) == 388, "BP dataset row count changed: review schema first"

if INCLUDE_RENJITH:
    github_dir = Path("/content/renjith_seven_segment_ocr")
    if not (github_dir / "data/ocr_training_data/gt.txt").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "--single-branch",
                        RENJITH_REPO_URL, str(github_dir)], check=True)
    training_root = github_dir / "data/ocr_training_data"
    gt_file = training_root / "gt.txt"
    if not gt_file.is_file():
        raise FileNotFoundError(f"Expected original recognition annotations: {gt_file}")
    accepted = 0
    total = 0
    for line_num, line in enumerate(gt_file.read_text(encoding="utf-8-sig").splitlines(), 1):
        if not line.strip():
            continue
        total += 1
        if "\t" not in line:
            extra_source_errors.append({"origin": "renjith", "row": line_num,
                                        "error": "Expected filename TAB transcription"})
            continue
        rel_name, full_text = line.split("\t", 1)
        target = (training_root / rel_name.strip()).resolve()
        if not target.is_relative_to(training_root.resolve()):
            extra_source_errors.append({"origin": "renjith", "row": line_num,
                                        "error": "Path escapes dataset folder"})
            continue
        try:
            with Image.open(target) as original:
                rgb = ImageOps.exif_transpose(original).convert("RGB")
                accepted += int(accept_external_image(
                    rgb, full_text, "renjith_ocr",
                    f"renjithsasidharan/seven-segment-ocr:{rel_name.strip()}", line_num))
        except Exception as exc:
            extra_source_errors.append({"origin": "renjith", "row": line_num,
                                        "source": rel_name, "error": str(exc)})
    extra_source_stats["renjith_ocr"] = {"provided": total, "accepted": accepted}

# Errors indicate corrupt/missing data or contradictory labels: stop rather than silently skip them.
dump_json(RUN_DIR / "additional_source_rejections.json", extra_source_rejections)
dump_json(RUN_DIR / "additional_source_errors.json", extra_source_errors)
dump_json(RUN_DIR / "additional_source_stats.json", extra_source_stats)
if extra_source_errors:
    print(json.dumps(extra_source_errors[:10], indent=2))
    raise ValueError(f"Review {len(extra_source_errors)} errors in additional OCR sources")
for name, details in extra_source_stats.items():
    print(name, details)
records = personal_records + public_records
dump_json(RUN_DIR / "public_manifest.json", public_records)
dump_json(RUN_DIR / "prepared_manifest.json", records)
dump_json(RUN_DIR / "duplicate_report.json", duplicate_report)
print("Total accepted external OCR rows:", len(public_records))
print("Additional source exclusions:", Counter(x["reason"] for x in extra_source_rejections))

## 7. Build the balanced training lists

Every accepted public image is used. Personal training rows are repeated just enough to reach the target personal fraction where needed; online augmentation makes their repeated presentations vary. These are repeated training entries, not additional independent images. Validation and test lists contain unique personal images only.

The second phase uses personal training images only. Neither phase can access personal validation/test examples through its training list. There is no image-concatenation augmentation.


In [ ]:
real_train = personal_splits["train"]
repeat_factor = max(1, math.ceil(
    len(public_records) * TARGET_PERSONAL_FRACTION /
    ((1 - TARGET_PERSONAL_FRACTION) * len(real_train))
))
mixed_train = real_train * repeat_factor + public_records
random.Random(SEED).shuffle(mixed_train)
list_paths = {
    "mixed_train": LIST_DIR / "mixed_train.txt",
    "personal_train": LIST_DIR / "personal_train.txt",
    "val": LIST_DIR / "personal_val.txt",
    "test": LIST_DIR / "personal_test.txt",
}
write_label_list(list_paths["mixed_train"], mixed_train)
write_label_list(list_paths["personal_train"], real_train)
write_label_list(list_paths["val"], personal_splits["val"])
write_label_list(list_paths["test"], personal_splits["test"])

train_hashes = {r["sha256"] for r in mixed_train}
val_hashes = {r["sha256"] for r in personal_splits["val"]}
test_hashes = {r["sha256"] for r in personal_splits["test"]}
assert not (train_hashes & val_hashes or train_hashes & test_hashes or val_hashes & test_hashes)
if PERSONAL_SPLIT_BY == "label":
    assert not ({r["label"] for r in mixed_train} & held_labels)
mixture_summary = {
    "personal_unique_train": len(real_train), "public_unique_train": len(public_records),
    "personal_repeat_factor": repeat_factor, "mixed_epoch_entries": len(mixed_train),
    "actual_personal_fraction": len(real_train) * repeat_factor / len(mixed_train),
    "personal_val": len(personal_splits["val"]), "personal_test": len(personal_splits["test"]),
}
dump_json(RUN_DIR / "mixture_summary.json", mixture_summary)
print(json.dumps(mixture_summary, indent=2))


## 8. Inspect both image sources

Titles show the original label and the actual training target. Under the default policy, dots are intentionally ignored and leading zeros are preserved. A filename label is not proof that a crop contains that value: inspect your own samples carefully. Full-device or multiple-display images need correct crops before training.


In [ ]:
import matplotlib.pyplot as plt

preview_groups = [("Personal train", real_train)]
for origin in ("7SEG_OCR", "bp_monitor_digits", "renjith_ocr"):
    subset = [r for r in public_records if r["origin"] == origin]
    if subset:
        preview_groups.append((origin, subset))
preview_groups.append(("Personal validation", personal_splits["val"]))
fig, axes = plt.subplots(len(preview_groups), 5,
                         figsize=(17, 2.7 * len(preview_groups)), squeeze=False)
for row_index, (title, group) in enumerate(preview_groups):
    samples = random.Random(SEED + row_index).sample(group, min(5, len(group)))
    for col_index, ax in enumerate(axes[row_index]):
        ax.axis("off")
        if col_index < len(samples):
            item = samples[col_index]
            ax.imshow(Image.open(PREPARED_DIR / item["path"]).convert("RGB"))
            ax.set_title(f"{title} | {item['original_label']} → {item['label']}", fontsize=8)
plt.tight_layout()
plt.show()

## 9. Conservative online augmentation

Training only: expanded-canvas rotation up to 1.5 degrees (20% of training presentations; only 1 degree during personal adaptation), small added margins, mild gamma/brightness, and at most one mild blur/noise/JPEG/downsampling operation. No flips, erasing, random crop, elastic distortion, or image concatenation. Validation and test are deterministic.

These changes preserve the readable digit sequence. Severe segment damage must not be simulated with an unchanged label. Set `AUGMENTATION_ENABLED=False` for a clean baseline using the same split.


In [ ]:
AUGMENTATION_SOURCE = 'import math\nimport random\nimport cv2\nimport numpy as np\n\n\nclass DigitDisplayAug:\n    """Conservative label-preserving augmentation for cropped display rows."""\n\n    def __init__(self, enabled=True, rotation_prob=0.25, max_rotation=2.0,\n                 padding_prob=0.15, photometric_prob=0.35,\n                 degradation_prob=0.25, **kwargs):\n        self.enabled = enabled\n        self.rotation_prob = rotation_prob\n        self.max_rotation = max_rotation\n        self.padding_prob = padding_prob\n        self.photometric_prob = photometric_prob\n        self.degradation_prob = degradation_prob\n\n    def __call__(self, data):\n        if not self.enabled:\n            return data\n        image = data["image"].copy()\n        height, width = image.shape[:2]\n        if random.random() < self.rotation_prob:\n            angle = random.uniform(-self.max_rotation, self.max_rotation)\n            matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)\n            cosine, sine = abs(matrix[0, 0]), abs(matrix[0, 1])\n            new_width = math.ceil(width * cosine + height * sine) + 2\n            new_height = math.ceil(height * cosine + width * sine) + 2\n            matrix[0, 2] += (new_width - width) / 2\n            matrix[1, 2] += (new_height - height) / 2\n            image = cv2.warpAffine(image, matrix, (new_width, new_height),\n                                   flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)\n        if random.random() < self.padding_prob:\n            height, width = image.shape[:2]\n            margin_y = max(1, round(height * random.uniform(0.01, 0.04)))\n            margin_x = max(1, round(width * random.uniform(0.005, 0.02)))\n            image = cv2.copyMakeBorder(image, margin_y, margin_y, margin_x, margin_x,\n                                      cv2.BORDER_REPLICATE)\n        if random.random() < self.photometric_prob:\n            gamma = random.uniform(0.85, 1.15)\n            gain = random.uniform(0.92, 1.08)\n            table = np.clip(((np.arange(256) / 255.0) ** gamma) * 255.0 * gain,\n                            0, 255).astype(np.uint8)\n            image = cv2.LUT(image, table)\n        if min(image.shape[:2]) >= 24 and random.random() < self.degradation_prob:\n            operation = random.choice(("blur", "noise", "jpeg", "downsample"))\n            if operation == "blur":\n                image = cv2.GaussianBlur(image, (3, 3), random.uniform(0.25, 0.55))\n            elif operation == "noise":\n                noise = np.random.normal(0, random.uniform(1.0, 3.0), image.shape)\n                image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)\n            elif operation == "jpeg":\n                success, encoded = cv2.imencode(".jpg", image,\n                                                [cv2.IMWRITE_JPEG_QUALITY, random.randint(80, 96)])\n                if success:\n                    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)\n                    if decoded is not None:\n                        image = decoded\n            else:\n                height, width = image.shape[:2]\n                scale = random.uniform(0.85, 0.97)\n                small = cv2.resize(image, (max(1, round(width * scale)),\n                                          max(1, round(height * scale))), interpolation=cv2.INTER_AREA)\n                image = cv2.resize(small, (width, height), interpolation=cv2.INTER_LINEAR)\n        data["image"] = np.ascontiguousarray(image)\n        return data\n'

augmentation_path = REPO_DIR / "ppocr/data/imaug/digit_display_aug.py"
augmentation_path.write_text(AUGMENTATION_SOURCE, encoding="utf-8")
(RUN_DIR / "digit_display_aug.py").write_text(AUGMENTATION_SOURCE, encoding="utf-8")
operator_init = REPO_DIR / "ppocr/data/imaug/__init__.py"
registration = "from .digit_display_aug import DigitDisplayAug"
init_source = operator_init.read_text(encoding="utf-8")
if registration not in init_source:
    operator_init.write_text(init_source + "\n" + registration + "\n", encoding="utf-8")

AUGMENTATION_SETTINGS = {
    "enabled": AUGMENTATION_ENABLED,
    "rotation_prob": 0.20,
    "max_rotation": 1.5,
    "padding_prob": 0.15,
    "photometric_prob": 0.35,
    "degradation_prob": 0.25,
}

import importlib.util
spec = importlib.util.spec_from_file_location("digit_display_aug_preview", augmentation_path)
aug_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(aug_module)
digit_augment = aug_module.DigitDisplayAug(**AUGMENTATION_SETTINGS)
print("Augmentation installed:", augmentation_path)

In [ ]:
import cv2

random_state, numpy_state = random.getstate(), np.random.get_state()
random.seed(SEED)
np.random.seed(SEED)
examples = random.sample([r for r in records if r["split"] == "train"], min(5, len(real_train)))
fig, axes = plt.subplots(len(examples), 5, figsize=(15, 2.4 * len(examples)), squeeze=False)
for row_index, item in enumerate(examples):
    image = np.asarray(Image.open(PREPARED_DIR / item["path"]).convert("RGB"))[:, :, ::-1].copy()
    for col in range(5):
        augmented = image if col == 0 else digit_augment({"image": image.copy(), "label": item["label"]})["image"]
        axes[row_index, col].imshow(cv2.cvtColor(augmented, cv2.COLOR_BGR2RGB))
        axes[row_index, col].set_title(("Original" if col == 0 else f"Augmented {col}") + f" | {item['label']}")
        axes[row_index, col].axis("off")
plt.tight_layout()
plt.show()
random.setstate(random_state)
np.random.set_state(numpy_state)

## 10. Build both training configurations

Mixed training starts from the official medium pretrained model. Its vocabulary-dependent output layers are reinitialized when their shapes differ; the pretrained visual backbone and compatible layers are still loaded.

The personal-adaptation phase starts from the mixed phase's best weights with a fresh optimizer and a smaller learning rate. Checkpoints are not used to accidentally resume epoch counters. Both phases use one shape, all rows (`ratio_list=[1.0]`), and the same personal validation set.


In [ ]:
import yaml

BASE_CONFIG = REPO_DIR / "configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml"
DICT_PATH = RUN_DIR / "digits.txt"
DICT_PATH.write_text("\n".join(CHARACTERS) + "\n", encoding="utf-8")
PRETRAINED_PATH = MODEL_CACHE / "PP-OCRv6_medium_rec_pretrained.pdparams"
PRETRAINED_URL = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams"
if not PRETRAINED_PATH.exists():
    download_atomic(PRETRAINED_URL, PRETRAINED_PATH)

def build_training_config(stage, train_list, entries, epochs, learning_rate, initial_weights):
    stage_directory = RUN_DIR / stage
    stage_directory.mkdir(exist_ok=True)
    cfg = yaml.safe_load(BASE_CONFIG.read_text())
    cfg["Global"].update({
        "model_name": "PP-OCRv6_medium_rec", "use_gpu": True, "distributed": False, "seed": SEED,
        "epoch_num": epochs, "save_model_dir": str(stage_directory), "save_epoch_step": 10,
        "eval_batch_step": [0, max(1, math.ceil(entries / BATCH_SIZE))], "print_batch_step": 10,
        "character_dict_path": str(DICT_PATH), "use_space_char": False,
        "max_text_length": MAX_TEXT_LENGTH, "d2s_train_image_shape": [3, IMAGE_HEIGHT, IMAGE_WIDTH],
        "pretrained_model": str(initial_weights), "checkpoints": None,
        "use_amp": False, "use_visualdl": False, "cal_metric_during_train": True,
    })
    cfg["Optimizer"]["lr"].update({
        "learning_rate": learning_rate, "warmup_epoch": min(3 if stage == "mixed" else 1, epochs),
    })
    transforms = [
        {"DecodeImage": {"img_mode": "BGR", "channel_first": False}},
        {"DigitDisplayAug": AUGMENTATION_SETTINGS.copy()},
        {"MultiLabelEncode": {"gtc_encode": "NRTRLabelEncode"}},
        {"RecResizeImg": {"image_shape": [3, IMAGE_HEIGHT, IMAGE_WIDTH], "padding": True}},
        {"KeepKeys": {"keep_keys": ["image", "label_ctc", "label_gtc", "length", "valid_ratio"]}},
    ]
    if stage == "personal":
        transforms[1]["DigitDisplayAug"].update({
            "rotation_prob": 0.12, "max_rotation": 1.0, "photometric_prob": 0.25, "degradation_prob": 0.15,
        })
    cfg["Train"]["dataset"] = {
        "name": "SimpleDataSet", "data_dir": str(PREPARED_DIR),
        "label_file_list": [str(train_list)], "ratio_list": [1.0], "transforms": transforms,
    }
    cfg["Train"].pop("sampler", None)
    cfg["Train"]["loader"] = {
        "shuffle": True, "drop_last": False, "num_workers": NUM_WORKERS,
        "batch_size_per_card": BATCH_SIZE,
    }
    cfg["Eval"]["dataset"] = {
        "name": "SimpleDataSet", "data_dir": str(PREPARED_DIR), "ratio_list": [1.0],
        "label_file_list": [str(list_paths["val"])],
        "transforms": [copy.deepcopy(op) for op in transforms if "DigitDisplayAug" not in op],
    }
    cfg["Eval"]["loader"] = {
        "shuffle": False, "drop_last": False, "num_workers": NUM_WORKERS,
        "batch_size_per_card": BATCH_SIZE,
    }
    destination = stage_directory / "train_config.yml"
    destination.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return cfg, destination

mixed_config, MIXED_CONFIG_PATH = build_training_config(
    "mixed", list_paths["mixed_train"], len(mixed_train), MIXED_EPOCHS,
    MIXED_LEARNING_RATE, PRETRAINED_PATH,
)
dump_json(RUN_DIR / "run_settings.json", {
    "model": "PP-OCRv6_medium_rec", "repo_tag": PADDLEOCR_TAG, "seed": SEED,
    "label_mode": PERSONAL_LABEL_MODE, "split_by": PERSONAL_SPLIT_BY,
    "label_policy": LABEL_POLICY, "characters": CHARACTERS,
    "fixed_crop_normalized": FIXED_CROP_NORMALIZED,
    "image_shape": [3, IMAGE_HEIGHT, IMAGE_WIDTH],
    "huggingface_repo": HF_REPO, "huggingface_revision": HF_REVISION,
    "additional_datasets": {"bp_monitor_digits": INCLUDE_BP_DIGITS,
                            "renjith_gt": INCLUDE_RENJITH},
    "augmentation": AUGMENTATION_SETTINGS, "mixture": mixture_summary,
})
print("Mixed-stage configuration:", MIXED_CONFIG_PATH)
print("If GPU memory is insufficient, lower BATCH_SIZE to 16 and rebuild the configurations.")

## 11. Validate the full data pipeline

The preflight runs every training and validation batch without model gradients. This checks augmentation registration, label encoding, finite tensors, complete row counts, and fixed shapes before expensive model training. All prepared images were independently decoded and validated earlier.


In [ ]:
PREFLIGHT_SOURCE = 'import sys\nimport copy\nimport yaml\nimport paddle\nfrom ppocr.data import build_dataloader\nfrom ppocr.utils.logging import get_logger\n\nconfig = yaml.safe_load(open(sys.argv[1], encoding="utf-8"))\nlogger = get_logger()\npaddle.set_device("cpu")\nfor mode in ("Train", "Eval"):\n    check = copy.deepcopy(config)\n    check[mode]["loader"]["num_workers"] = 0\n    loader = build_dataloader(check, mode, paddle.CPUPlace(), logger)\n    expected = sum(1 for line in open(check[mode]["dataset"]["label_file_list"][0]) if line.strip())\n    assert len(loader.dataset) == expected, (mode, len(loader.dataset), expected)\n    total = 0\n    for index, batch in enumerate(loader):\n        assert len(batch) == 5\n        assert list(batch[0].shape[1:]) == config["Global"]["d2s_train_image_shape"]\n        assert bool(paddle.isfinite(batch[0]).all())\n        total += batch[0].shape[0]\n    assert total == expected, (total, expected)\n    print(mode, "pipeline OK; records:", expected, "checked:", total)\n'
PREFLIGHT_PATH = REPO_DIR / "check_digit_pipeline.py"
PREFLIGHT_PATH.write_text(PREFLIGHT_SOURCE, encoding="utf-8")
subprocess.run([sys.executable, str(PREFLIGHT_PATH), str(MIXED_CONFIG_PATH)], cwd=REPO_DIR, check=True)

## 12. Train the mixed model

No additional data-audit assertion needs to be toggled. Read the previews before starting this expensive cell. The default run uses 35 mixed epochs followed by 15 personal epochs. Validation selects checkpoints; the final epoch is not automatically treated as the best.

Data and configuration are snapshotted for this run. Rerunning a completed training cell is blocked to prevent overwriting its checkpoints. Start a new run for a new experiment.


In [ ]:
def train_stage(config_path):
    cfg = yaml.safe_load(Path(config_path).read_text())
    stage_dir = Path(cfg["Global"]["save_model_dir"])
    best = stage_dir / "best_accuracy"
    if any(stage_dir.glob("*.pdparams")):
        raise FileExistsError(f"Stage already has checkpoints: {stage_dir}. Start a new run instead of overwriting.")
    log_path = stage_dir / "train.log"
    environment = os.environ.copy()
    environment.update({"PYTHONUNBUFFERED": "1", "FLAGS_allocator_strategy": "auto_growth"})
    command = [sys.executable, "-u", "tools/train.py", "-c", str(config_path)]
    process = subprocess.Popen(command, cwd=REPO_DIR, env=environment, text=True, bufsize=1,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    try:
        with log_path.open("w", encoding="utf-8") as stream:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        if process.stdout:
            process.stdout.close()
    if process.wait() != 0:
        raise RuntimeError(f"Training failed. See {log_path}.")
    if not Path(str(best) + ".pdparams").is_file():
        raise FileNotFoundError(f"No best_accuracy checkpoint was created in {stage_dir}.")
    return best

MIXED_BEST = train_stage(MIXED_CONFIG_PATH)
print("Best mixed checkpoint:", MIXED_BEST)

## 13. Adapt on personal training images

This stage uses your training images only, including valid zero readings. It starts from the mixed checkpoint, not the original multilingual weights, and resets the optimizer intentionally. A separate personal-stage best checkpoint is preserved. Set `PERSONAL_ADAPT_EPOCHS=0` before configuration to skip this phase.


In [ ]:
from pathlib import Path

root = Path(mixed_config["Global"]["save_model_dir"])

for name in ["best_accuracy", "latest"]:
    for ext in [".pdparams", ".pdopt", ".states"]:
        path = root / f"{name}{ext}"
        print(
            path.name,
            "FOUND" if path.exists() else "MISSING"
        )

In [ ]:
from pathlib import Path

MIXED_BEST = Path(mixed_config["Global"]["save_model_dir"]) / "best_accuracy"

assert Path(str(MIXED_BEST) + ".pdparams").is_file(), \
    "Best model checkpoint not found!"

PERSONAL_ADAPT_EPOCHS = 0

candidates = [
    {
        "stage": "mixed",
        "checkpoint": str(MIXED_BEST),
        "config": str(MIXED_CONFIG_PATH)
    }
]

dump_json(
    RUN_DIR / "candidate_checkpoints.json",
    candidates
)

print("Best model successfully recovered!")
print("Checkpoint:", MIXED_BEST)
print("Personal fine-tuning skipped.")
print("Ready for evaluation.")

In [ ]:
candidates = [{"stage": "mixed", "checkpoint": str(MIXED_BEST), "config": str(MIXED_CONFIG_PATH)}]
if PERSONAL_ADAPT_EPOCHS:
    personal_config, PERSONAL_CONFIG_PATH = build_training_config(
        "personal", list_paths["personal_train"], len(real_train), PERSONAL_ADAPT_EPOCHS,
        PERSONAL_LEARNING_RATE, MIXED_BEST,
    )
    subprocess.run([sys.executable, str(PREFLIGHT_PATH), str(PERSONAL_CONFIG_PATH)], cwd=REPO_DIR, check=True)
    PERSONAL_BEST = train_stage(PERSONAL_CONFIG_PATH)
    candidates.append({"stage": "personal", "checkpoint": str(PERSONAL_BEST), "config": str(PERSONAL_CONFIG_PATH)})
dump_json(RUN_DIR / "candidate_checkpoints.json", candidates)

## 14. Prediction utility and validation-based final selection

Both stage-best checkpoints are evaluated on the same personal validation images. The winner has higher exact-string accuracy; ties use lower CER, then prefer the mixed stage. Personal test labels are not consulted. The final selected weights are copied to the run root as `best_accuracy.pdparams` for deployment and future fine-tuning.

The prediction utility loads the network once, checks checkpoint tensor shapes, and uses BGR, aspect-preserving fixed resize, right padding, and [-1, 1] normalization. Decoder confidence is not a calibrated correctness probability.


In [ ]:
PREDICTION_SOURCE = 'import argparse\nimport copy\nimport csv\nimport json\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nimport paddle\nimport yaml\nfrom ppocr.modeling.architectures import build_model\nfrom ppocr.postprocess import build_post_process\nfrom ppocr.data.imaug.rec_img_aug import resize_norm_img\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    parser.add_argument("--checkpoint", required=True)\n    parser.add_argument("--items", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--batch-size", type=int, default=32)\n    parser.add_argument("--device", default="gpu")\n    args = parser.parse_args()\n    paddle.set_device(args.device)\n    config = yaml.safe_load(Path(args.config).read_text())\n    decoder = build_post_process(config["PostProcess"], config["Global"])\n    character_count = len(decoder.character)\n    architecture = copy.deepcopy(config["Architecture"])\n    architecture["Head"]["out_channels_list"] = {\n        "CTCLabelDecode": character_count,\n        "NRTRLabelDecode": character_count + 3,\n        "SARLabelDecode": character_count + 2,\n    }\n    model = build_model(architecture)\n    checkpoint = args.checkpoint\n    checkpoint = checkpoint if checkpoint.endswith(".pdparams") else checkpoint + ".pdparams"\n    state = paddle.load(checkpoint)\n    expected = model.state_dict()\n    mismatches = [name for name, value in expected.items()\n                  if name not in state or tuple(value.shape) != tuple(state[name].shape)]\n    if mismatches:\n        raise ValueError(f"Incompatible checkpoint tensors: {mismatches[:10]}")\n    model.set_state_dict(state)\n    model.eval()\n    items = json.loads(Path(args.items).read_text())\n    shape = config["Global"]["d2s_train_image_shape"]\n    rows = []\n    with paddle.no_grad():\n        for offset in range(0, len(items), args.batch_size):\n            subset = items[offset:offset + args.batch_size]\n            tensors = []\n            for item in subset:\n                data = np.frombuffer(Path(item["path"]).read_bytes(), dtype=np.uint8)\n                image = cv2.imdecode(data, cv2.IMREAD_COLOR)\n                if image is None:\n                    raise ValueError(f"Cannot decode {item[\'path\']}")\n                if image.shape[1] / image.shape[0] > shape[2] / shape[1]:\n                    raise ValueError(f"Input is too wide for the configured resize: {item[\'path\']}")\n                tensor, _ = resize_norm_img(image, shape, padding=True)\n                tensors.append(tensor)\n            predictions = decoder(model(paddle.to_tensor(np.stack(tensors))))\n            for item, prediction in zip(subset, predictions):\n                text, confidence = prediction\n                rows.append({"path": item["path"], "source": item.get("source", ""),\n                             "label": item.get("label", ""), "prediction": text,\n                             "confidence": float(confidence)})\n    with Path(args.output).open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=["path", "source", "label", "prediction", "confidence"])\n        writer.writeheader()\n        writer.writerows(rows)\n    print(f"Saved {len(rows)} predictions to {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n'
PREDICT_SCRIPT = REPO_DIR / "predict_digits.py"
PREDICT_SCRIPT.write_text(PREDICTION_SOURCE, encoding="utf-8")
(RUN_DIR / "predict_digits.py").write_text(PREDICTION_SOURCE, encoding="utf-8")

In [ ]:
from rapidfuzz.distance import Levenshtein

def prediction_job(items, checkpoint, config_path, job_name):
    items_path = RUN_DIR / f"{job_name}_items.json"
    output_path = RUN_DIR / f"{job_name}_predictions.csv"
    dump_json(items_path, items)
    subprocess.run([sys.executable, str(PREDICT_SCRIPT), "--config", str(config_path),
                    "--checkpoint", str(checkpoint), "--items", str(items_path),
                    "--output", str(output_path), "--batch-size", str(BATCH_SIZE)], cwd=REPO_DIR, check=True)
    with output_path.open(encoding="utf-8", newline="") as stream:
        result_rows = list(csv.DictReader(stream))
    if len(result_rows) != len(items):
        raise ValueError("Prediction count differs from the input count.")
    return result_rows

def metric_summary(result_rows):
    if not result_rows:
        raise ValueError("Cannot evaluate an empty set.")
    correct = sum(r["label"] == r["prediction"] for r in result_rows)
    edits = sum(Levenshtein.distance(r["label"], r["prediction"]) for r in result_rows)
    characters = sum(len(r["label"]) for r in result_rows)
    zero_rows = [r for r in result_rows if set(r["label"]) == {"0"}]
    return {"samples": len(result_rows), "exact_accuracy": correct / len(result_rows),
            "cer": edits / characters, "zero_samples": len(zero_rows),
            "zero_exact_accuracy": (sum(r["label"] == r["prediction"] for r in zero_rows) / len(zero_rows)
                                    if zero_rows else None)}

validation_items = [{"path": str(PREPARED_DIR / r["path"]), "source": r["source"], "label": r["label"]}
                    for r in personal_splits["val"]]
candidate_predictions = {}
for candidate in candidates:
    predictions = prediction_job(validation_items, candidate["checkpoint"], candidate["config"],
                                 f"validation_{candidate['stage']}")
    candidate_predictions[candidate["stage"]] = predictions
    candidate["validation"] = metric_summary(predictions)
selected = sorted(candidates, key=lambda c: (-c["validation"]["exact_accuracy"],
                                             c["validation"]["cer"], c["stage"] != "mixed"))[0]
BEST_CHECKPOINT = RUN_DIR / "best_accuracy"
shutil.copy2(selected["checkpoint"] + ".pdparams", str(BEST_CHECKPOINT) + ".pdparams")
config = yaml.safe_load(Path(selected["config"]).read_text())
CONFIG_PATH = RUN_DIR / "train_config.yml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
dump_json(RUN_DIR / "model_selection.json", {"selected_stage": selected["stage"], "candidates": candidates})
print(json.dumps({"selected_stage": selected["stage"], "validation": selected["validation"]}, indent=2))
print("Deployment checkpoint:", BEST_CHECKPOINT)

## 15. Learning curves and final personal test

The validation curves compare the two stages. Training metrics include augmented images and are smoothed. Final test metrics measure exact sequence accuracy, CER, and zero-only reading accuracy if zero-only examples exist in the held-out test split. No test examples are added back to training.


In [ ]:
plt.figure(figsize=(11, 3))
for candidate in candidates:
    text = (RUN_DIR / candidate["stage"] / "train.log").read_text(encoding="utf-8")
    values = [float(value) for value in re.findall(r"cur metric, acc: ([0-9.eE+-]+)", text)]
    plt.plot(range(1, len(values) + 1), np.asarray(values) * 100, marker=".", label=candidate["stage"])
plt.xlabel("Validation evaluation within stage")
plt.ylabel("Exact accuracy (%)")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

test_items = [{"path": str(PREPARED_DIR / r["path"]), "source": r["source"], "label": r["label"]}
              for r in personal_splits["test"]]
test_predictions = prediction_job(test_items, BEST_CHECKPOINT, CONFIG_PATH, "personal_test")
test_summary = metric_summary(test_predictions)
dump_json(RUN_DIR / "test_summary.json", test_summary)
print(json.dumps(test_summary, indent=2))
errors = [row.copy() for row in test_predictions if row["label"] != row["prediction"]]
for row in errors:
    row["error_type"] = ("shorter" if len(row["prediction"]) < len(row["label"]) else
                         "longer" if len(row["prediction"]) > len(row["label"]) else "same_length")
with (RUN_DIR / "test_errors.csv").open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=["path", "source", "label", "prediction", "confidence", "error_type"])
    writer.writeheader()
    writer.writerows(errors)
shown = sorted(errors, key=lambda row: float(row["confidence"]), reverse=True)[:12]
if shown:
    fig, axes = plt.subplots(math.ceil(len(shown) / 3), 3, figsize=(14, 3 * math.ceil(len(shown) / 3)), squeeze=False)
    for axis, row in zip(axes.flat, shown):
        with Image.open(row["path"]) as image:
            axis.imshow(image)
        axis.set_title(f"True: {row['label']} | Pred: {row['prediction']}\nScore: {float(row['confidence']):.3f}")
        axis.axis("off")
    for axis in axes.flat[len(shown):]:
        axis.axis("off")
    plt.tight_layout()
    plt.show()

## 16. Export at the actual input height

PaddleOCR v3.7.0 hard-codes height 48 for this architecture in its static export specification. This cell corrects that specification and the TensorRT metadata to **the actual training height** before export. No manual source edit is needed. It then verifies the exported preprocessing shape and character dictionary.

Only the selected checkpoint is exported. Export output stays in local Colab storage, not Drive. Do not combine a new model's weights with an old model's `inference.yml`.


In [ ]:
export_source_path = REPO_DIR / "ppocr/utils/export_model.py"
export_source = export_source_path.read_text(encoding="utf-8")
old_spec = "shape=[None, 3, 48, -1]"
new_spec = f"shape=[None, 3, {IMAGE_HEIGHT}, -1]"
if old_spec not in export_source and new_spec not in export_source:
    raise RuntimeError("The expected SVTR_LCNet export input specification was not found.")
export_source = export_source.replace(old_spec, new_spec)
export_source = export_source.replace(
    "[[1, 3, 48, 160], [1, 3, 48, 320], [8, 3, 48, 3200]]",
    f"[[1, 3, {IMAGE_HEIGHT}, 160], [1, 3, {IMAGE_HEIGHT}, 320], [8, 3, {IMAGE_HEIGHT}, 3200]]",
)
export_source_path.write_text(export_source, encoding="utf-8")
EXPORT_DIR = LOCAL_DIR / "deployment/inference"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_config = copy.deepcopy(config)
export_config["Global"].update({"checkpoints": None, "pretrained_model": str(BEST_CHECKPOINT),
                                 "save_inference_dir": str(EXPORT_DIR)})
EXPORT_CONFIG_PATH = EXPORT_DIR.parent / "export_config.yml"
EXPORT_CONFIG_PATH.write_text(yaml.safe_dump(export_config, sort_keys=False), encoding="utf-8")
subprocess.run([sys.executable, "tools/export_model.py", "-c", str(EXPORT_CONFIG_PATH)],
               cwd=REPO_DIR, check=True)
shutil.copy2(DICT_PATH, EXPORT_DIR / "digits.txt")
infer_config = yaml.safe_load((EXPORT_DIR / "inference.yml").read_text())
exported_shape = next(op["RecResizeImg"]["image_shape"] for op in infer_config["PreProcess"]["transform_ops"]
                      if "RecResizeImg" in op)
assert exported_shape == [3, IMAGE_HEIGHT, IMAGE_WIDTH]
assert infer_config["Global"]["model_name"] == "PP-OCRv6_medium_rec"
assert infer_config["PostProcess"]["character_dict"] == list(CHARACTERS)
for name in ["inference.json", "inference.pdiparams", "inference.yml"]:
    assert (EXPORT_DIR / name).is_file(), name
print("Exported shape:", exported_shape)
print("Exported size:", f"{sum(p.stat().st_size for p in EXPORT_DIR.rglob('*') if p.is_file()) / 1024**2:.2f} MiB")

## 17. Verify the exported model against the selected dynamic checkpoint

The standalone static inference utility runs with Paddle Inference on CPU and the same fixed preprocessing. This test checks actual graph execution at height 64, not just the YAML metadata. A small deterministic subset of **validation** is compared with the selected checkpoint. A mismatch stops packaging so an inconsistent deployment is not silently shipped.


In [ ]:
STATIC_PREDICTION_SOURCE = 'import argparse\nimport csv\nimport json\nimport math\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nimport paddle.inference as inference\nimport yaml\n\n\ndef prepare_image(path, shape):\n    image = cv2.imdecode(np.frombuffer(Path(path).read_bytes(), dtype=np.uint8), cv2.IMREAD_COLOR)\n    if image is None:\n        raise ValueError(f"Cannot decode {path}")\n    channels, height, width = shape\n    source_height, source_width = image.shape[:2]\n    required = math.ceil(source_width * height / width)\n    if required > source_height:\n        extra = required - source_height\n        image = cv2.copyMakeBorder(image, extra // 2, extra - extra // 2, 0, 0, cv2.BORDER_REPLICATE)\n    resized_width = min(width, math.ceil(height * image.shape[1] / image.shape[0]))\n    resized = cv2.resize(image, (resized_width, height)).astype(np.float32).transpose(2, 0, 1)\n    tensor = np.zeros((channels, height, width), dtype=np.float32)\n    tensor[:, :, :resized_width] = (resized / 255.0 - 0.5) / 0.5\n    return tensor\n\n\ndef decode_ctc(probabilities, characters):\n    vocabulary = ["blank"] + list(characters)\n    index = probabilities.argmax(axis=-1)\n    probability = probabilities.max(axis=-1)\n    result = []\n    for indices, scores in zip(index, probability):\n        keep = np.ones(len(indices), dtype=bool)\n        keep[1:] = indices[1:] != indices[:-1]\n        keep &= indices != 0\n        text = "".join(vocabulary[int(i)] for i in indices[keep])\n        confidence = float(scores[keep].mean()) if keep.any() else 0.0\n        result.append((text, confidence))\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--model-dir", required=True)\n    parser.add_argument("--items", required=True)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args()\n    directory = Path(args.model_dir)\n    cfg = yaml.safe_load((directory / "inference.yml").read_text())\n    shape = next(op["RecResizeImg"]["image_shape"] for op in cfg["PreProcess"]["transform_ops"]\n                 if "RecResizeImg" in op)\n    characters = cfg["PostProcess"]["character_dict"]\n    settings = inference.Config(str(directory / "inference.json"), str(directory / "inference.pdiparams"))\n    settings.disable_gpu()\n    settings.disable_mkldnn()\n    settings.set_cpu_math_library_num_threads(2)\n    predictor = inference.create_predictor(settings)\n    input_handle = predictor.get_input_handle(predictor.get_input_names()[0])\n    declared = input_handle.shape()\n    if len(declared) == 4 and declared[2] not in (-1, shape[1]):\n        raise ValueError(f"Graph input height {declared[2]} differs from preprocessing height {shape[1]}")\n    output_handle = predictor.get_output_handle(predictor.get_output_names()[0])\n    items = json.loads(Path(args.items).read_text())\n    rows = []\n    for item in items:\n        tensor = prepare_image(item["path"], shape)[None, ...]\n        input_handle.reshape(tensor.shape)\n        input_handle.copy_from_cpu(tensor)\n        predictor.run()\n        probabilities = output_handle.copy_to_cpu()\n        if probabilities.shape[-1] != len(characters) + 1:\n            raise ValueError("The static output vocabulary size is inconsistent.")\n        text, score = decode_ctc(probabilities, characters)[0]\n        rows.append({"path": item["path"], "source": item.get("source", ""),\n                     "label": item.get("label", ""), "prediction": text, "confidence": score})\n    with Path(args.output).open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=["path", "source", "label", "prediction", "confidence"])\n        writer.writeheader()\n        writer.writerows(rows)\n    print(f"Static inference OK: {len(rows)} images, input shape {shape}")\n\n\nif __name__ == "__main__":\n    main()\n'
STATIC_SCRIPT = EXPORT_DIR.parent / "predict_exported.py"
STATIC_SCRIPT.write_text(STATIC_PREDICTION_SOURCE, encoding="utf-8")
export_check_items = validation_items[:min(16, len(validation_items))]
EXPORT_CHECK_ITEMS = EXPORT_DIR.parent / "export_check_items.json"
EXPORT_CHECK_OUTPUT = EXPORT_DIR.parent / "export_check_predictions.csv"
dump_json(EXPORT_CHECK_ITEMS, export_check_items)
subprocess.run([sys.executable, str(STATIC_SCRIPT), "--model-dir", str(EXPORT_DIR),
                "--items", str(EXPORT_CHECK_ITEMS), "--output", str(EXPORT_CHECK_OUTPUT)], check=True)
with EXPORT_CHECK_OUTPUT.open(encoding="utf-8", newline="") as stream:
    static_rows = list(csv.DictReader(stream))
dynamic_rows = candidate_predictions[selected["stage"]][:len(export_check_items)]
assert len(static_rows) == len(dynamic_rows)
export_differences = [{"path": a["path"], "dynamic": b["prediction"], "static": a["prediction"]}
                      for a, b in zip(static_rows, dynamic_rows) if a["prediction"] != b["prediction"]]
dump_json(RUN_DIR / "export_validation.json", {"checked": len(static_rows), "differences": export_differences})
if export_differences:
    raise RuntimeError(f"Static/dynamic predictions differ: {export_differences}. Inspect before deployment.")
print("Export matches selected checkpoint on", len(static_rows), "validation images.")


## 18. Optional new-image prediction

Upload cropped display-row images. Your fixed normalized crop is applied when configured. Filenames are identifiers, not ground truth. The model is loaded once for the uploaded batch. A per-image crop in the training manifest cannot be guessed for a new upload.


In [ ]:
from google.colab import files

PREDICT_NEW_IMAGES = True
if PREDICT_NEW_IMAGES:
    uploaded = files.upload()
    upload_directory = LOCAL_DIR / "new_images"
    upload_directory.mkdir(exist_ok=True)
    items = []
    for index, (name, payload) in enumerate(uploaded.items()):
        with Image.open(io.BytesIO(payload)) as original:
            image = personal_crop(ImageOps.exif_transpose(original).convert("RGB"), {})
        image, _ = fit_input_aspect(image)
        path = upload_directory / f"{index:05d}.png"
        image.save(path)
        items.append({"path": str(path), "source": name})
    if not items:
        raise ValueError("No images were uploaded.")
    results = prediction_job(items, BEST_CHECKPOINT, CONFIG_PATH, "uploaded")
    from IPython.display import display
    for row in results:
        display(Image.open(row["path"]))
        print(row["source"], "->", row["prediction"], "score:", row["confidence"])

## 19. Download the deployment package

The ZIP is created only in temporary `/content` storage and downloaded to your browser automatically. It is not written to Google Drive. The training checkpoints/reports remain in Drive separately.

The package contains the selected weights, exported inference files, the exact vocabulary, both prediction utilities, configuration, model-selection/test reports, source provenance, and augmentation source. It excludes training images and optimizer files.

For your project, copy the **contents of `inference/`** into a new model directory, set `PADDLE_MODEL_DIR` to that directory, use model name `PP-OCRv6_medium_rec`, and convert PIL RGB arrays to contiguous BGR arrays before passing them to `TextRecognition`. Keep the exported `inference.yml`; do not reuse the small model's configuration. Its resize shape is [3, 64, 320] by default. Do not set the `TextRecognition(input_shape=...)` override: in PaddleX 3.7.2 it stretches the image rather than using the trained padding policy.


In [ ]:
attribution = (
    "External training source: MiXaiLL76/7SEG_OCR\n"
    "Dataset URL: https://huggingface.co/datasets/MiXaiLL76/7SEG_OCR\n"
    f"Pinned revision: {HF_REVISION}\n"
    "Declared dataset license: MIT; see included source README.\n"
    f"Label policy: {LABEL_POLICY}. Public examples are training-only.\n"
    "Additional source: https://huggingface.co/datasets/chakibmed/bp-monitor-seven-segment (MIT; digit crops only)\n"
    "Additional source: https://github.com/renjithsasidharan/seven-segment-ocr (check repository license)\n"
    "This package does not redistribute the source dataset images.\n"
    "Personal images and labels remain the responsibility of their owner.\n"
)
(EXPORT_DIR.parent / "DATA_SOURCES.txt").write_text(attribution, encoding="utf-8")
PACKAGE_PATH = LOCAL_DIR / "medium_digit_ocr_package.zip"
selected_files = {
    "best_accuracy.pdparams", "train_config.yml", "digits.txt", "predict_digits.py",
    "digit_display_aug.py", "run_settings.json", "mixture_summary.json", "model_selection.json",
    "test_summary.json", "test_errors.csv", "personal_test_predictions.csv", "export_validation.json",
    "7SEG_OCR_README.md", "public_rejections.json", "duplicate_report.json",
    "personal_manifest.json", "public_manifest.json", "prepared_manifest.json",
    "additional_source_rejections.json", "additional_source_errors.json",
    "additional_source_stats.json",
}
with zipfile.ZipFile(PACKAGE_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EXPORT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, Path("inference") / path.relative_to(EXPORT_DIR))
    for name in sorted(selected_files):
        path = RUN_DIR / name
        if path.is_file():
            archive.write(path, name)
    for name in ["predict_exported.py", "export_config.yml", "DATA_SOURCES.txt"]:
        archive.write(EXPORT_DIR.parent / name, name)
    archive.write(SPLIT_STATE_PATH, "personal_split_assignments.json")
print("Downloaded model package:", PACKAGE_PATH.name)
print("Selected stage:", selected["stage"])
print("Personal test metrics:", json.dumps(test_summary, indent=2))
files.download(str(PACKAGE_PATH))

## Interpreting the result

- More synthetic images do not guarantee improved real-camera accuracy. Compare against a personal-only baseline with the identical split.
- Label policy, input crops, and real device conditions must match deployment. Preserve zero-only strings and leading zeros.
- Validation chooses both the best epoch and the final stage. Test is for final reporting, not repeated hyperparameter tuning.
- Similar video frames can leak across splits unless you supply meaningful session/device groups. Exact duplicate removal alone cannot prevent that.
- Public label exclusions are intentional under unseen-number splitting. Do not remove them merely to increase training counts.
- For future fine-tuning, use `best_accuracy.pdparams` and the same vocabulary/configuration. Do not attempt to load small-model weights into medium.

### Sources
- [7SEG_OCR dataset card and declared MIT license](https://huggingface.co/datasets/MiXaiLL76/7SEG_OCR)
- [Pinned dataset revision](https://huggingface.co/datasets/MiXaiLL76/7SEG_OCR/tree/59873e636d5bd8bd850b3ba5788ee761cef5e895)
- [Official PaddleOCR v3.7.0 medium training configuration](https://github.com/PaddlePaddle/PaddleOCR/blob/v3.7.0/configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml)
- [Official export implementation](https://github.com/PaddlePaddle/PaddleOCR/blob/v3.7.0/ppocr/utils/export_model.py)
